# HighCap vs ELITE — Interactive Market Screener

Two orthogonal strategies compared side-by-side, with tunable universe filters.

- **HighCap** — pure heuristic: rank the diverse universe by ACS `cap_rate_proxy` and take top-N. **Cash-flow strategy.** Best hit-rate (~22%) with zero ML.
- **ELITE** — analog search restricted to the top X% of historical top-decile winners. Ranks candidates by cosine similarity to those elite exemplars. **Appreciation strategy.** ~21% hit-rate.

These two methods are **essentially uncorrelated** — for 100-300K, they overlap less than random chance. You have to pick a thesis (or blend both).

## How to use this notebook

1. Edit the config in section 0 (population band, HHI ceiling, top-N, max per state, ELITE percentile)
2. Run all cells top-to-bottom
3. Section 3 shows today's picks side-by-side + intersection
4. Section 4 shows historical hit-rate validation

**All cells reuse `backtest/analog_search.py` — restart kernel if you edit that module.**

## 0. Configuration knobs — edit these

In [17]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '/Users/premputtegowda/reps/reps-tracker/emerging_re')

import duckdb, numpy as np, pandas as pd
from pathlib import Path

WAREHOUSE = "/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb"

# ⚙️ TUNE ME — universe filter
POP_MIN, POP_MAX     = 100_000, 500_000      # try (100_000, 300_000) for tertiary
DIVERSITY_HHI_MAX    = 0.10                  # 0.10 = well-diversified; lower = stricter
                                             # Reference: 0.15 loose, 0.10 tight, 0.08 very tight

# ⚙️ TUNE ME — picking parameters
TOP_N                = 10                    # picks per method (5 = focused, 10 = broader)
MAX_PER_STATE        = 4                     # state cap for geographic diversification
ELITE_TOP_PCT        = 0.25                  # ELITE filters to top X% of winners
                                             # 0.10 = ~12 elite winners (sharp, volatile)
                                             # 0.25 = ~30 elite winners (broader, stable)
                                             # 0.50 = ~60 winners (much like baseline)

# ⚙️ Fixed (don't usually change)
HORIZON_YEARS        = 3
TARGET               = 'total_return'        # or 'rent'

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 100)

print(f"Universe: pop {POP_MIN//1000}K-{POP_MAX//1000}K   HHI ≤ {DIVERSITY_HHI_MAX}")
print(f"Picks: top-{TOP_N} per method   max {MAX_PER_STATE} per state   ELITE = top {ELITE_TOP_PCT*100:.0f}% of winners")

Universe: pop 100K-500K   HHI ≤ 0.1
Picks: top-10 per method   max 4 per state   ELITE = top 25% of winners


## 1. Universe overview

Confirm the filter yields a reasonable universe size per cohort (>50 counties is comfortable for top-decile winners).

In [18]:
con = duckdb.connect(WAREHOUSE, read_only=True)
uni = con.execute(f'''
    SELECT EXTRACT(YEAR FROM period_end_date)::INT AS year,
           COUNT(*) AS n,
           AVG(cap_rate_proxy)*100 AS avg_cap_pct,
           AVG(employment_diversity_hhi) AS avg_hhi
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
      AND pop_latest > {POP_MIN} AND pop_latest <= {POP_MAX}
      AND employment_diversity_hhi IS NOT NULL
      AND employment_diversity_hhi <= {DIVERSITY_HHI_MAX}
    GROUP BY 1 ORDER BY 1
''').df()
con.close()
uni['avg_cap_pct'] = uni['avg_cap_pct'].round(2)
uni['avg_hhi'] = uni['avg_hhi'].round(4)
print(f"Universe size per cohort (2015-2025):")
print(uni[uni['year'] >= 2015].to_string(index=False))

Universe size per cohort (2015-2025):
 year   n  avg_cap_pct  avg_hhi
 2015 421         6.34   0.0756
 2016 420         6.13   0.0759
 2017 421         6.00   0.0762
 2018 419         5.87   0.0764
 2019 418         5.83   0.0762
 2020 414         5.72   0.0756
 2021 414         5.75   0.0763
 2022 412         5.26   0.0763
 2023 415         5.02   0.0766
 2024 414         5.07   0.0766
 2025 419         5.00   0.0757


## 2. Run both methods

Loads the panel once, computes features, then runs both methods on the current universe.

In [19]:
from backtest.analog_search import (
    load_tertiary_panel, load_mf_context, build_acs_target,
    build_feature_matrix, zscore_within_cohort, walk_forward,
    TARGET_COL, TARGET_COL_TOTAL_RETURN,
    REALIZED_TOTAL_ZORI36_COL, REALIZED_RETURN_COL_ZORI36,
)

target_col = TARGET_COL_TOTAL_RETURN if TARGET == 'total_return' else TARGET_COL

# Load panel + features once
panel = load_tertiary_panel(pop_min=POP_MIN, pop_max=POP_MAX, hhi_max=DIVERSITY_HHI_MAX)
panel = panel.merge(build_acs_target(horizon_years=HORIZON_YEARS), on=["fips","period_end_date"], how="left")
panel = panel.merge(load_mf_context(), on="fips", how="left")
panel_fe, feature_cols = build_feature_matrix(panel)
panel_z = zscore_within_cohort(panel_fe, feature_cols)
print(f"Panel: {len(panel):,} rows, {len(feature_cols)} features")

# Run ELITE (analog search with elite_top_pct filter, strict PIT for honest walk-forward)
print(f"\n### Running ELITE (top {ELITE_TOP_PCT*100:.0f}% of winners) ###")
res_elite = walk_forward(panel_z, panel_fe, feature_cols,
    target_col=target_col, strict_pit_targets=True,
    top_n=TOP_N, max_per_state=MAX_PER_STATE, elite_top_pct=ELITE_TOP_PCT)
print(f"  {len(res_elite)} test-year iterations completed")

Panel: 10,847 rows, 91 features

### Running ELITE (top 25% of winners) ###

Walk-forward schedule [STRICT-PIT ELITE(25%)] (21 iterations, min_train_years=5, target=target_total_return_36mo_demeaned):
  SKIP test=2005   train_cohorts= 0   span=(none)
  SKIP test=2006   train_cohorts= 0   span=(none)
  SKIP test=2007   train_cohorts= 0   span=(none)
  SKIP test=2008   train_cohorts= 0   span=(none)
  SKIP test=2009   train_cohorts= 0   span=(none)
  SKIP test=2010   train_cohorts= 0   span=(none)
  SKIP test=2011   train_cohorts= 1   span=2008-2008
  SKIP test=2012   train_cohorts= 2   span=2008-2009
  SKIP test=2013   train_cohorts= 3   span=2008-2010
  SKIP test=2014   train_cohorts= 4   span=2008-2011
  test=2015   train_cohorts= 5   span=2008-2012
  test=2016   train_cohorts= 6   span=2008-2013
  test=2017   train_cohorts= 7   span=2008-2014
  test=2018   train_cohorts= 8   span=2008-2015
  test=2019   train_cohorts= 9   span=2008-2016
  test=2020   train_cohorts=10   span=2008-2017

In [20]:
# HighCap: pure heuristic — no training needed
def highcap_picks_for_year(year, universe_df, top_n, max_per_state):
    c = universe_df[universe_df['year'] == year].dropna(subset=['cap_rate_proxy'])
    if len(c) < 20:
        return pd.DataFrame()
    ranked = c.sort_values('cap_rate_proxy', ascending=False)
    picked, state_counts = [], {}
    for _, r in ranked.iterrows():
        s = r['state_name']
        if state_counts.get(s, 0) >= max_per_state:
            continue
        picked.append(r)
        state_counts[s] = state_counts.get(s, 0) + 1
        if len(picked) >= top_n:
            break
    return pd.DataFrame(picked)

# Load the full universe for HighCap (with realized returns for hit-rate calc later)
con = duckdb.connect(WAREHOUSE, read_only=True)
uni_full = con.execute(f'''
    SELECT fips, county_name, state_name, pop_latest,
           EXTRACT(YEAR FROM period_end_date)::INT AS year,
           cap_rate_proxy, mf_cap_rate_estimated,
           employment_diversity_hhi,
           zori_forward_return_36mo
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
      AND pop_latest > {POP_MIN} AND pop_latest <= {POP_MAX}
      AND employment_diversity_hhi IS NOT NULL
      AND employment_diversity_hhi <= {DIVERSITY_HHI_MAX}
''').df()
con.close()

# Attach 3yr exit cap for realized TR calc
exit_lookup = uni_full[['fips','year','cap_rate_proxy']].rename(columns={'year':'ey','cap_rate_proxy':'cap_exit'})
uni_full['ey'] = uni_full['year'] + 3
uni_full = uni_full.merge(exit_lookup, on=['fips','ey'], how='left').drop(columns=['ey'])
uni_full['tr_zori36'] = (1 + uni_full['zori_forward_return_36mo']) * (uni_full['cap_rate_proxy'] / uni_full['cap_exit']) - 1

print(f"HighCap universe: {len(uni_full):,} rows loaded")

HighCap universe: 10,847 rows loaded


## 3. Today's picks — 2025 cohort (side-by-side + intersection)

The core output. Shows both methods' top-N picks with cap rates, plus the intersection (markets both methods endorse).

In [21]:
year_to_show = 2025

# ELITE picks for the target year
elite_result = next((r for r in res_elite if r.test_year == year_to_show), None)
if elite_result is None:
    raise ValueError(f"No ELITE result for {year_to_show}")
elite_picks = elite_result.top5[['fips','county_name','state_name','pop_latest']].copy()

# HighCap picks for the target year
hc_picks = highcap_picks_for_year(year_to_show, uni_full, TOP_N, MAX_PER_STATE)

# Attach cap rates from mart (not z-scored)
def enrich_with_caps(picks_df, uni_df, year):
    caps = uni_df[uni_df['year']==year][['fips','cap_rate_proxy','mf_cap_rate_estimated','employment_diversity_hhi']]
    return picks_df.merge(caps, on='fips', how='left')

hc_picks = enrich_with_caps(hc_picks, uni_full, year_to_show)
elite_picks = enrich_with_caps(elite_picks, uni_full, year_to_show)

print("=" * 130)
print(f"{year_to_show} TOP-{TOP_N} — pop {POP_MIN//1000}K-{POP_MAX//1000}K + HHI ≤ {DIVERSITY_HHI_MAX}")
print("=" * 130)
print(f"\n{'#':<3} {'HighCap':<40} {'cap_ACS':>8} {'cap_MF':>8}  {'HHI':>7}  |  {'ELITE':<40} {'cap_ACS':>8} {'cap_MF':>8}  {'HHI':>7}")
print("-" * 130)
for i in range(TOP_N):
    def fmt(r):
        if r is None: return " "*68
        name = f"{r['county_name'].replace(' County','')}, {r['state_name']}"
        mf_str = f"{(r['mf_cap_rate_estimated'] or 0)*100:>6.2f}%" if pd.notna(r['mf_cap_rate_estimated']) else "   n/a "
        return f"{name:<40} {r['cap_rate_proxy']*100:>7.2f}% {mf_str}   {r['employment_diversity_hhi']:>7.4f}"
    hc_r = hc_picks.iloc[i] if i < len(hc_picks) else None
    el_r = elite_picks.iloc[i] if i < len(elite_picks) else None
    print(f"{i+1:<3} {fmt(hc_r)}  |  {fmt(el_r)}")

2025 TOP-10 — pop 100K-500K + HHI ≤ 0.1

#   HighCap                                   cap_ACS   cap_MF      HHI  |  ELITE                                     cap_ACS   cap_MF      HHI
----------------------------------------------------------------------------------------------------------------------------------


KeyError: 'mf_cap_rate_estimated'

In [22]:
# Intersection: markets appearing in BOTH lists
hc_set = set(hc_picks['fips'])
el_set = set(elite_picks['fips'])
inter = hc_set & el_set
union = hc_set | el_set

print("=" * 100)
print(f"⭐ INTERSECTION — {year_to_show} markets in both HighCap AND ELITE top-{TOP_N}")
print("=" * 100)
if inter:
    for f in inter:
        r = uni_full[(uni_full['year']==year_to_show) & (uni_full['fips']==f)].iloc[0]
        mf = (r['mf_cap_rate_estimated'] or 0)*100
        print(f"  ⭐ {r['county_name']}, {r['state_name']:20s}  pop={int(r['pop_latest']):>7,}  "
              f"cap_ACS={r['cap_rate_proxy']*100:5.2f}%  cap_MF={mf:5.2f}%  HHI={r['employment_diversity_hhi']:.4f}")
else:
    print("  (No consensus picks at this top-N)")
    print(f"  Try larger TOP_N — the two methods are largely orthogonal, so small N often yields no overlap.")

print(f"\nOverlap: {len(inter)}/{TOP_N}   Jaccard: {len(inter)}/{len(union)} = {len(inter)/len(union) if union else 0:.2f}")
n_universe = len(uni_full[uni_full['year']==year_to_show])
random_expected = TOP_N * TOP_N / n_universe
print(f"Random baseline: {random_expected:.2f} expected overlap (universe n={n_universe})")

⭐ INTERSECTION — 2025 markets in both HighCap AND ELITE top-10
  (No consensus picks at this top-N)
  Try larger TOP_N — the two methods are largely orthogonal, so small N often yields no overlap.

Overlap: 0/10   Jaccard: 0/20 = 0.00
Random baseline: 0.24 expected overlap (universe n=419)


In [23]:
# Intersection scaling — how much overlap at broader N?
print("How does intersection grow as we broaden the pick set?\n")
print(f"{'top-N':>8}  {'HighCap top-N':>15}  {'ELITE top-N':>15}  {'intersection':>14}  {'random baseline':>17}")

hc_ranked = uni_full[uni_full['year']==year_to_show].dropna(subset=['cap_rate_proxy']).sort_values('cap_rate_proxy', ascending=False)['fips'].tolist()
el_ranked = elite_picks['fips'].tolist()   # already in rank order

for N in [5, 10, 15, 20, 30]:
    hc_topN = hc_ranked[:N]
    el_topN = el_ranked[:N] if len(el_ranked) >= N else el_ranked
    n_int = len(set(hc_topN) & set(el_topN))
    exp = N * N / n_universe
    marker = " ⭐" if n_int > exp * 1.5 else ("   " if n_int >= exp * 0.5 else " ✗ ")
    print(f"{N:>8}  {len(hc_topN):>15d}  {len(el_topN):>15d}  {n_int:>14d}{marker}  {exp:>17.2f}")

print("\nLegend: ⭐ = above random    ✗ = below random")

How does intersection grow as we broaden the pick set?

   top-N    HighCap top-N      ELITE top-N    intersection    random baseline
       5                5                5               0 ✗                0.06
      10               10               10               0 ✗                0.24
      15               15               10               0 ✗                0.54
      20               20               10               1                  0.95
      30               30               10               1 ✗                2.15

Legend: ⭐ = above random    ✗ = below random


## 4. Historical hit-rate validation

For each observable cohort (2015-2022), how many of each method's top-10 picks actually ended up in the ACTUAL top-10 by realized 36mo total return?

In [24]:
def actual_top_n_fips(year, uni_full, n):
    c = uni_full[(uni_full['year']==year) & uni_full['tr_zori36'].notna()]
    if c.empty: return None
    return set(c.nlargest(n, 'tr_zori36')['fips'].tolist())

hit_rows = []
for r in res_elite:
    yr = r.test_year
    actual = actual_top_n_fips(yr, uni_full, TOP_N)
    if actual is None:
        continue
    elite_set = set(r.top5['fips'].tolist())
    hc_picks_yr = highcap_picks_for_year(yr, uni_full, TOP_N, MAX_PER_STATE)
    hc_set_yr = set(hc_picks_yr['fips'].tolist())

    hit_rows.append({
        'year': yr,
        'universe_size': len(uni_full[uni_full['year']==yr]),
        'HighCap_hits': len(hc_set_yr & actual),
        'ELITE_hits': len(elite_set & actual),
        'intersection_hits': len((hc_set_yr & elite_set) & actual),
        'intersection_size': len(hc_set_yr & elite_set),
    })

hits = pd.DataFrame(hit_rows)
print(f"Precision-at-{TOP_N}: how many picks actually landed in the ACTUAL top-{TOP_N} by realized TR-ZORI36")
print("=" * 100)
print(hits.to_string(index=False))

obs = hits.dropna(subset=['HighCap_hits','ELITE_hits'])
if len(obs):
    n_avg = obs['universe_size'].mean()
    random_exp = TOP_N * TOP_N / n_avg
    print(f"\n--- Aggregate over {len(obs)} observable cohorts (avg universe n={n_avg:.0f}) ---")
    print(f"  HighCap:      avg {obs['HighCap_hits'].mean():.2f}/{TOP_N} hits ({obs['HighCap_hits'].mean()/TOP_N*100:.1f}% precision)")
    print(f"  ELITE:        avg {obs['ELITE_hits'].mean():.2f}/{TOP_N} hits ({obs['ELITE_hits'].mean()/TOP_N*100:.1f}% precision)")
    print(f"  Intersection: avg {obs['intersection_hits'].mean():.2f} hits, avg {obs['intersection_size'].mean():.1f} consensus picks/year")
    print(f"  Random baseline: {random_exp:.2f} expected hits ({random_exp/TOP_N*100:.1f}% precision)")

Precision-at-10: how many picks actually landed in the ACTUAL top-10 by realized TR-ZORI36
 year  universe_size  HighCap_hits  ELITE_hits  intersection_hits  intersection_size
 2015            421             0           0                  0                  0
 2016            420             0           1                  0                  0
 2017            421             2           3                  0                  0
 2018            419             1           2                  0                  0
 2019            418             0           2                  0                  0
 2020            414             2           0                  0                  0
 2021            414             2           0                  0                  0
 2022            412             1           0                  0                  0

--- Aggregate over 8 observable cohorts (avg universe n=417) ---
  HighCap:      avg 1.00/10 hits (10.0% precision)
  ELITE:        avg 1.00/10

## 5. Historical picks per cohort — both methods side-by-side

Full picks per cohort year (2015-2025) for both methods, plus consensus markers.

In [25]:
all_years = sorted([r.test_year for r in res_elite])

for yr in all_years:
    if yr < 2015:
        continue
    elite_r = next((r for r in res_elite if r.test_year == yr), None)
    if elite_r is None: continue
    elite_fips = elite_r.top5['fips'].tolist()
    hc_picks_yr = highcap_picks_for_year(yr, uni_full, TOP_N, MAX_PER_STATE)
    if hc_picks_yr.empty:
        continue
    hc_fips = hc_picks_yr['fips'].tolist()
    inter = set(hc_fips) & set(elite_fips)

    # Get realized top-N (if observable)
    actual = actual_top_n_fips(yr, uni_full, TOP_N)

    print(f"\n===== {yr} — top {TOP_N} =====")
    print(f"{'#':<3}  {'HighCap':<32}  {'ELITE':<32}  {'notes':<20}")
    for i in range(TOP_N):
        hc_r = hc_picks_yr.iloc[i] if i < len(hc_picks_yr) else None
        el_r = elite_r.top5.iloc[i] if i < len(elite_r.top5) else None
        hc_name = f"{hc_r['county_name'].replace(' County','')}, {hc_r['state_name'][:2]}" if hc_r is not None else ""
        el_name = f"{el_r['county_name'].replace(' County','')}, {el_r['state_name'][:2]}" if el_r is not None else ""
        notes = []
        if hc_r is not None and el_r is not None and hc_r['fips'] == el_r['fips']:
            notes.append('same rank')
        elif hc_r is not None and hc_r['fips'] in inter:
            notes.append('in both⭐')
        if actual is not None:
            if hc_r is not None and hc_r['fips'] in actual:
                notes.append('hc✓actual')
            if el_r is not None and el_r['fips'] in actual:
                notes.append('el✓actual')
        print(f"{i+1:<3}  {hc_name:<32}  {el_name:<32}  {' '.join(notes)}")

    if inter:
        inter_names = [uni_full[(uni_full['year']==yr) & (uni_full['fips']==f)].iloc[0]['county_name'].replace(' County','') for f in inter]
        print(f"   INTERSECTION: {', '.join(inter_names)}")


===== 2015 — top 10 =====
#    HighCap                           ELITE                             notes               
1    Robeson, No                       Eaton, Mi                         
2    Ector, Te                         Comal, Te                         
3    Liberty, Te                       Forsyth, Ge                       
4    Wyandotte, Ka                     Cherokee, Ge                      
5    Wichita, Te                       Sussex, De                        
6    Highlands, Fl                     Wilson, Te                        
7    Potter, Te                        Ascension Parish, Lo              
8    St. Lucie, Fl                     Rockwall, Te                      
9    Genesee, Mi                       Sumner, Te                        
10   St. Lawrence, Ne                  Gallatin, Mo                      

===== 2016 — top 10 =====
#    HighCap                           ELITE                             notes               
1    Wyandotte, Ka

## 6. Strategic interpretation

**HighCap picks** — high-yield markets. Older secondary metros with structural cap-rate premiums.
- Bet: collect the yield, mild appreciation if rates fall
- Best in: rising-yield era markets, cash-flow investors
- Historical hit-rate: ~22% precision

**ELITE picks** — growth-story markets. Feature profiles resembling past exceptional-return exemplars.
- Bet: appreciation via rent growth + cap compression
- Best in: growing suburbs adjacent to major metros, kickers with unique catalysts
- Historical hit-rate: ~21% precision

**Consensus picks** — markets endorsed by both methods.
- Rare (often zero at top-10 for 100-300K)
- When they occur, they're markets with **high current cap AND feature match to elite winners** — the exceptional combination of yield + growth-signal
- Highest conviction

**No consensus?** That's a real result. It means you must pick a thesis:
1. Higher HHI ceiling → broader universe but less-diverse markets
2. Wider TOP_N → more picks in each list, more chance of overlap
3. Different ELITE percentile → 0.10 (very tight) to 0.50 (very loose)
4. Different pop band → the two universes have quite different opportunity profiles

## Common tunings to try

```python
# Cash-flow deployment (tertiary, more picks, broader)
POP_MIN, POP_MAX = 100_000, 300_000
TOP_N = 15
ELITE_TOP_PCT = 0.25

# Sharpshooter (mid-size, focused, tightest ELITE filter)
POP_MIN, POP_MAX = 300_000, 500_000
TOP_N = 5
ELITE_TOP_PCT = 0.10

# Diversification / consensus-hunting (broad ELITE filter to increase overlap)
POP_MIN, POP_MAX = 200_000, 500_000
TOP_N = 20
ELITE_TOP_PCT = 0.50
```